# RGB-Agent Kaggle Competition Submission

This notebook prepares a Kaggle competition submission using the official `ARC-AGI-3-Agents` repository while reusing the planning logic from this `RGB-Agent` project.

Workflow:
- install ARC wheels and this project
- optionally install `vLLM` from an offline Kaggle dataset
- optionally start a local OpenAI-compatible model server
- write an ARC-compatible `RGBSubmissionAgent` into the official repo
- smoke test on a public game in offline mode
- run the official repo in `COMPETITION` mode

Important:
- for submission, do not rely on an external workstation server
- if you use `vLLM`, run it inside the Kaggle runtime
- package weights and Python wheels as Kaggle input datasets ahead of time


In [ ]:
from pathlib import Path
import os

PROJECT_CANDIDATES = [
    Path('/kaggle/working/RGB-Agent'),
    Path('/kaggle/input/rgb-agent/RGB-Agent'),
    Path('/kaggle/input/RGB-Agent/RGB-Agent'),
    Path.cwd(),
]

PROJECT_ROOT = None
for candidate in PROJECT_CANDIDATES:
    if (candidate / 'pyproject.toml').exists() and (candidate / 'rgb_agent').exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find the RGB-Agent project root.')

COMP_ROOT = Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-3')
ARC_WHEEL_DIR = COMP_ROOT / 'arc_agi_3_wheels'
ARC_AGENT_REPO_SRC = COMP_ROOT / 'ARC-AGI-3-Agents'
PUBLIC_ENV_DIR = COMP_ROOT / 'environment_files'
WORK_REPO = Path('/kaggle/working/ARC-AGI-3-Agents')

print('PROJECT_ROOT      =', PROJECT_ROOT)
print('COMP_ROOT         =', COMP_ROOT)
print('ARC_WHEEL_DIR     =', ARC_WHEEL_DIR)
print('ARC_AGENT_REPO_SRC=', ARC_AGENT_REPO_SRC)
print('PUBLIC_ENV_DIR    =', PUBLIC_ENV_DIR)


In [ ]:
import subprocess
import sys

if not ARC_WHEEL_DIR.exists():
    raise FileNotFoundError(f'ARC wheel directory not found: {ARC_WHEEL_DIR}')
if not ARC_AGENT_REPO_SRC.exists():
    raise FileNotFoundError(f'Official ARC repo not found: {ARC_AGENT_REPO_SRC}')

subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    '--no-index',
    f'--find-links={ARC_WHEEL_DIR}',
    'arc-agi',
    'python-dotenv',
    'requests',
])

subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    str(PROJECT_ROOT),
])

print('Base dependencies installed.')


## Configuration

Change these values for your packaged model and wheel datasets.

Typical submission pattern:
- upload model weights as a Kaggle dataset
- upload `vLLM` wheels as a Kaggle dataset
- use `Qwen/Qwen2.5-72B-Instruct-AWQ` weights for a more realistic single-GPU setup
- set `INSTALL_VLLM=True` and `START_VLLM=True`
- run a smoke test first
- finally set `RUN_COMPETITION=True`


In [ ]:
import os

INSTALL_VLLM = True
START_VLLM = True
RUN_SMOKE_TEST = False
RUN_COMPETITION = False

# Replace these with the actual dataset paths you attach in Kaggle.
VLLM_WHEELS_DIR = Path('/kaggle/input/your-vllm-wheels-dataset')
MODEL_DIR = Path('/kaggle/input/your-qwen25-72b-awq-model-dataset')
SERVED_MODEL_NAME = 'Qwen/Qwen2.5-72B-Instruct-AWQ'

RGB_AGENT_NAME = 'rgbsubmission'
RGB_MODEL = f'openai/{SERVED_MODEL_NAME}'
RGB_ANALYZER_BACKEND = 'direct'
RGB_ANALYZER_INTERVAL = 10
RGB_ANALYZER_RETRIES = 5
RGB_MAX_ACTIONS = 500
RGB_LOG_DIR = '/kaggle/working/rgb_submission_logs'

# Conservative vLLM defaults for a single 96GB RTX PRO 6000 Blackwell.
VLLM_HOST = '127.0.0.1'
VLLM_PORT = 8000
VLLM_GPU_MEMORY_UTILIZATION = 0.92
VLLM_MAX_MODEL_LEN = 16384
VLLM_MAX_NUM_SEQS = 8
VLLM_TENSOR_PARALLEL_SIZE = 1

OPENAI_BASE_URL = 'http://127.0.0.1:8000/v1'
OPENAI_API_KEY = 'EMPTY'

SMOKE_TEST_GAME = 'ls20'

print('MODEL_DIR         =', MODEL_DIR)
print('SERVED_MODEL_NAME =', SERVED_MODEL_NAME)
print('RGB_MODEL         =', RGB_MODEL)


In [ ]:
import subprocess
import sys

if INSTALL_VLLM:
    if not VLLM_WHEELS_DIR.exists():
        raise FileNotFoundError(f'vLLM wheels directory not found: {VLLM_WHEELS_DIR}')
    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        '--no-index',
        f'--find-links={VLLM_WHEELS_DIR}',
        'vllm',
        'transformers',
        'accelerate',
    ])
    print('vLLM dependencies installed.')
else:
    print('Skipping vLLM installation. Set INSTALL_VLLM=True if needed.')


In [ ]:
import requests
import subprocess
import sys
import time

vllm_proc = None

if START_VLLM:
    if not MODEL_DIR.exists():
        raise FileNotFoundError(f'Model directory not found: {MODEL_DIR}')

    cmd = [
        'vllm',
        'serve',
        str(MODEL_DIR),
        '--served-model-name', SERVED_MODEL_NAME,
        '--host', VLLM_HOST,
        '--port', str(VLLM_PORT),
        '--gpu-memory-utilization', str(VLLM_GPU_MEMORY_UTILIZATION),
        '--max-model-len', str(VLLM_MAX_MODEL_LEN),
        '--max-num-seqs', str(VLLM_MAX_NUM_SEQS),
        '--tensor-parallel-size', str(VLLM_TENSOR_PARALLEL_SIZE),
        '--generation-config', 'vllm',
        '--trust-remote-code',
    ]
    vllm_proc = subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)

    ready = False
    for _ in range(180):
        try:
            r = requests.get(f'http://{VLLM_HOST}:{VLLM_PORT}/v1/models', timeout=1)
            if r.ok:
                ready = True
                break
        except Exception:
            time.sleep(1)

    if not ready:
        raise RuntimeError('vLLM server did not become ready in time.')
    print('vLLM is ready.')
else:
    print('Skipping vLLM start. Set START_VLLM=True if you want a local model server.')


In [ ]:
import shutil
import textwrap

from rgb_agent.submission import build_submission_agent_source

if WORK_REPO.exists():
    shutil.rmtree(WORK_REPO)
shutil.copytree(ARC_AGENT_REPO_SRC, WORK_REPO)

agent_src = build_submission_agent_source('RGBSubmissionAgent')
agent_path = WORK_REPO / 'agents' / 'rgb_submission_agent.py'
agent_path.write_text(agent_src, encoding='utf-8')

agents_init = textwrap.dedent('''
from typing import Type
from dotenv import load_dotenv

from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .rgb_submission_agent import RGBSubmissionAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    "random": Random,
    "rgbsubmission": RGBSubmissionAgent,
}
''').strip() + '\n'
(WORK_REPO / 'agents' / '__init__.py').write_text(agents_init, encoding='utf-8')

print('Official ARC agent repo prepared at', WORK_REPO)
print('Custom agent file:', agent_path)


In [ ]:
import os
import subprocess
import sys

def build_env(operation_mode: str) -> dict:
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    env['OPENAI_BASE_URL'] = OPENAI_BASE_URL
    env['OPENAI_API_KEY'] = OPENAI_API_KEY
    env['RGB_AGENT_NAME'] = RGB_AGENT_NAME
    env['RGB_MODEL'] = RGB_MODEL
    env['RGB_ANALYZER_BACKEND'] = RGB_ANALYZER_BACKEND
    env['RGB_ANALYZER_INTERVAL'] = str(RGB_ANALYZER_INTERVAL)
    env['RGB_ANALYZER_RETRIES'] = str(RGB_ANALYZER_RETRIES)
    env['RGB_MAX_ACTIONS'] = str(RGB_MAX_ACTIONS)
    env['RGB_LOG_DIR'] = RGB_LOG_DIR
    env['OPERATION_MODE'] = operation_mode
    if PUBLIC_ENV_DIR.exists():
        env['ENVIRONMENTS_DIR'] = str(PUBLIC_ENV_DIR)
    return env

if RUN_SMOKE_TEST:
    cmd = [sys.executable, 'main.py', '--agent', 'rgbsubmission', '--game', SMOKE_TEST_GAME]
    print('Running smoke test:', ' '.join(cmd))
    subprocess.run(cmd, cwd=WORK_REPO, env=build_env('OFFLINE'), check=False)
else:
    print('Skipping smoke test. Set RUN_SMOKE_TEST=True to run one public game offline.')


In [ ]:
import subprocess
import sys

if RUN_COMPETITION:
    cmd = [sys.executable, 'main.py', '--agent', 'rgbsubmission']
    print('Starting competition run:', ' '.join(cmd))
    subprocess.run(cmd, cwd=WORK_REPO, env=build_env('COMPETITION'), check=False)
else:
    print('Competition run is disabled. Set RUN_COMPETITION=True for the final submission pass.')
